# Initialization and Reproducibility

The computational environment for the experiments is initialized, including the required libraries, working directories, execution device, and random seeds to ensure reproducibility.

In [ ]:
import os
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import TensorDataset, DataLoader
from nflows import transforms, distributions, flows

In [ ]:
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
# Define main paths
DATA_BASE = Path("/MUON-SPECTRUM-CNF/processed") # Adjust paths as needed
WORK_BASE = Path("/MUON-SPECTRUM-CNF/results") # Adjust paths as needed

MODEL_FINAL_DIR = WORK_BASE / "Model_Final"
VAL_DIR = MODEL_FINAL_DIR / "val"
TEST_DIR = MODEL_FINAL_DIR / "test"
VAL_IMG_DIR = VAL_DIR / "Images"
TEST_IMG_DIR = TEST_DIR / "Images"

MODEL_FINAL_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)
VAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
TEST_IMG_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = WORK_BASE / "Model_Final.zip"

# Select available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Define base seed
BASE_SEED = 42
os.environ["PYTHONHASHSEED"] = str(BASE_SEED)


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def reset_for_trial(base_seed: int = BASE_SEED):
    seed_everything(base_seed)


seed_everything(BASE_SEED)

# Configure CuDNN behavior
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = False

Device: cuda


# Site Definition and Data Loading

The training, validation, and test sites are established; each site is associated with its corresponding input file, and the datasets are loaded into unified DataFrame structures. A site label is added to each row to preserve the origin of every event after concatenation.

In [ ]:
from src.preprocessing import (
    load_sites,
    downsample_per_site_joint,
    make_loader,
)

In [ ]:
# Define cities 
train_files = {
    "lisb":  "lisb_3600.csv",   # h=0
    "std":   "std_3600.csv",    # h=14
    "iqa":   "iqa_3600.csv",    # h=34
    "lim":   "lim_3600.csv",    # h=150
    "chia":  "chia_3600.csv",   # h=522
    "truj":  "truj_3600.csv",   # h=560
    "bga":   "bga_3600.csv",    # h=956

    "bra":   "bra_3600.csv",    # h=1172
    "alb":   "alb_3600.csv",    # h=1619
    "mge":   "mge_3600.csv",    # h=1450   
    "denv":  "denv_3600.csv",   # h=1609
    "erz":   "erz_3600.csv",    # h=1890

    "sha":   "sha_3600.csv",    # h=2070
    "sudb":  "sudb_3600.csv",   # h=2100
    "pra":   "pra_3600.csv",    # h=2143  
    "pue":   "pue_3600.csv",    # h=2143  
    "pamp":  "pamp_3600.csv",   # h=2342
    "add":   "add_3600.csv",    # h=2355
    "asp":   "asp_3600.csv",    # h=2405
    "casp":  "casp_3600.csv",   # h=2600
    "srm":   "srm_3600.csv",    # h=2750
    "quie":  "quie_3600.csv",   # h=2850

    "taw":   "taw_3600.csv",    # h=3000
    "fair":  "fair_3600.csv",   # h=3034
    "ker":   "ker_3600.csv",    # h=3150
    "asm":   "asm_3600.csv",    # h=3250
    "ber":   "ber_3600.csv",    # h=3450
    "lpb":   "lpb_3600.csv",    # h=3630

    "and":   "and_3600.csv",    # h=4200
    "mks":   "mks_3600.csv",    # h=4205
    "tib":   "tib_3600.csv",    # h=4500
    "sng":   "sng_3600.csv",    # h=4550
    "kom":   "kom_3600.csv",    # h=4587
    "khun":  "khun_3600.csv",   # h=4700
    "cha":   "cha_3600.csv",    # h=5230
}

val_files = {
    "lond":  "lond_3600.csv",   # h=24
    "gua":   "gua_3600.csv",    # h=1490
    "serb":  "serb_3600.csv",   # h=2750 
    "ima":   "ima_3600.csv",    # h=4600
    "ata":   "ata_3600.csv",    # h=5105
}

test_files = {
    "sjs":   "sjs_3600.csv",    # h=1172
    "shi":   "shi_3600.csv",    # h=1500
    "cuchu": "cuchu_3600.csv",  # h=3250
    "jun":   "jun_3600.csv",    # h=3454
    "zard":  "zard_3600.csv",   # h=4221
}

train_sites = list(train_files.keys())
val_sites = list(val_files.keys())
test_sites = list(test_files.keys())

print("Train sites:", train_sites)
print("Val sites:", val_sites)
print("Test sites:", test_sites)

In [ ]:
# Load training and validation data
df_train = load_sites(train_sites, train_files, DATA_BASE)
df_val   = load_sites(val_sites, val_files, DATA_BASE)

print("Raw train rows:", len(df_train))
print("Raw val rows:", len(df_val))

# Data preprocessing

The main preprocessing settings are defined, and the data preparation pipeline used before training is implemented. First, a joint per-site downsampling strategy is applied to reduce site imbalance while preserving the joint distribution of energy and zenith angle.

In [5]:
# Data preprocessing settings
TARGET_PER_SITE_TRAIN = 370_000
N_BINS = 100
EPS = 1e-12
EPS_ANG = 1e-6
context_cols = ["h", "bx", "bz"]
RNG = np.random.default_rng(BASE_SEED)

In [ ]:
# Global energy bins from raw training data
logE_train_all = np.log(df_train["energy"].to_numpy(dtype=float))
LogEmin = logE_train_all.min()
LogEmax = logE_train_all.max()
binsE = np.linspace(LogEmin, LogEmax + 1e-8, N_BINS + 1)

# Angle bins in [0, 90]
binsA = np.linspace(0.0, 90.0 + 1e-8, N_BINS + 1)

# Apply joint downsampling to the training set
df_train = downsample_per_site_joint(
    df_train,
    TARGET_PER_SITE_TRAIN,
    binsE,
    binsA,
    RNG
)

print("Train after joint downsample:", len(df_train))
print(df_train["site"].value_counts().head())

The conditional variables are normalized with a MinMax transformation computed from the training set, and the same statistics are later applied to the validation set. Energy is first transformed to logarithmic scale and then normalized. For the angular component, the variable 𝜇=1−cos(𝜃).

In [ ]:
stats = {}

# MinMax normalization for context variables
for col in context_cols:
    x = df_train[col].to_numpy(dtype=float)
    x_min = float(x.min())
    x_max = float(x.max())
    x_range = x_max - x_min

    df_train[col + "_mm"] = (df_train[col] - x_min) / (x_range + EPS)
    df_val[col + "_mm"]   = (df_val[col]   - x_min) / (x_range + EPS)

    stats[col] = {"min": x_min, "max": x_max, "range": x_range}

# Log + MinMax normalization for energy
logE_train = np.log(df_train["energy"].to_numpy(dtype=float))
z_min = float(logE_train.min())
z_max = float(logE_train.max())
z_range = z_max - z_min

df_train["logE_mm"] = (logE_train - z_min) / (z_range + EPS)
stats["logE"] = {"min": z_min, "max": z_max, "range": z_range}

# Apply training statistics to validation data
logE_val = np.log(df_val["energy"].to_numpy(dtype=float))
df_val["logE_mm"] = (logE_val - z_min) / (z_range + EPS)

# Angular variable for the model
mu_train = 1.0 - np.cos(np.deg2rad(df_train["theta"].to_numpy(dtype=float)))
mu_val   = 1.0 - np.cos(np.deg2rad(df_val["theta"].to_numpy(dtype=float)))

df_train["mu"] = np.clip(mu_train, EPS_ANG, 1.0 - EPS_ANG)
df_val["mu"]   = np.clip(mu_val,   EPS_ANG, 1.0 - EPS_ANG)

stats["mu"] = {"min": 0.0, "max": 1.0, "range": 1.0}

stats

In [ ]:
# Training tensors
X_train = torch.from_numpy(df_train[["logE_mm", "mu"]].to_numpy(dtype=np.float32))
C_train = torch.from_numpy(df_train[[c + "_mm" for c in context_cols]].to_numpy(dtype=np.float32))
train_tensor = TensorDataset(X_train, C_train)

# Validation tensors
X_val = torch.from_numpy(df_val[["logE_mm", "mu"]].to_numpy(dtype=np.float32))
C_val = torch.from_numpy(df_val[[c + "_mm" for c in context_cols]].to_numpy(dtype=np.float32))
val_tensor = TensorDataset(X_val, C_val)

# CNFs architecture and training

The fixed training hyperparameters, the conditional flow architecture, and the optimization procedure are established. The model is constructed as a two-dimensional conditional normalizing flow, where the target variables are the normalized logarithmic energy and the transformed angular variable μ. Training is performed by minimizing the negative log-likelihood over the training set, while the validation loss is evaluated after each epoch.

In [9]:
# Training configuration
EPOCHS = 20
LR_FIXED = 3e-4

# Hyperparameters
FINAL_HIDDEN_FEATURES = 192
FINAL_NUM_LAYERS = 6
FINAL_NUM_BINS = 4
FINAL_TAIL_BOUND = 8

In [ ]:
from src.model import (
    build_flow,
    val_flow,
    train_flow,
)

# Validation utilities and inference helpers

The necessary tools for validation and inference are integrated, including the forward and inverse transformations, the spectral error metrics used to compare the model predictions with the reference distributions.

In [12]:
# Validation settings
Y_MIN = 0.0
Y_MAX = 0.85
MU_MIN = 1e-6
MU_MAX = 1.0 - 1e-6
SAMPLE_BATCH = 8192

# Histogram bins and evaluation seeds
bins_E = np.logspace(-2, 4, 50)
bins_theta = np.linspace(0.0, 90, 91)
EVAL_SEEDS = [5, 503, 1001, 1501, 1999]

In [ ]:
from src.validation_helpers import (
    logE_to_mm,
    mm_to_energy,
    add_mu,
    mu_to_theta,
    con_mm,
    energy_spectrum,
    angle_spectrum,
    abs_relative_error_per_bin,
    weighted_relative_error_per_bin,
    masked_mean,
    range_mean,
    sample_model_joint,
)

In [ ]:
def scalar_abs_relative_error(real_value, model_value, eps=1e-12):
    """
    Compute the absolute relative error between two scalar values.
    """
    real_value = float(real_value)
    model_value = float(model_value)

    if not np.isfinite(real_value) or not np.isfinite(model_value):
        return float("nan")
    if real_value <= eps:
        return float("nan")

    return float(abs((model_value - real_value) / real_value))

A function is defined to estimate the expected muon flux from the site altitude, and another function is used to convert that flux into an expected number of events for a given time interval. These utilities provide the event-count estimate used during evaluation.

In [15]:
def expected_flux_from_height(h):
    """
    Estimate the expected flux in part/(m²·s) from the site altitude.
    """
    return 101 * np.exp(0.0002109 * float(h))


def expected_events_from_height(h, seconds=3600.0):
    """
    Convert the altitude-based flux estimate into an expected number of events.
    """
    n_events = int(round(expected_flux_from_height(h) * float(seconds)))
    return max(n_events, 1)

Although normalization was calculated from the entire training set, individual sites do not necessarily cover the entire interval [0,1]; therefore, sampling was restricted to y ≤ 0.86, which covers the region where most of the site-level data is concentrated.

# Plotting utilities for spectral comparison


The plotting style adopted for validation was established, along with the functions used to save the comparisons of the energy and angular spectra. Each figure includes the reference spectrum, the mean model prediction, the associated ±1σ band across validation seeds, and the contribution of the weighted relative error shown in a lower panel.

In [17]:
# Global plotting style
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 13,
    "lines.linewidth": 2.5,
    "grid.linewidth": 0.9,
})

In [ ]:
from src.plotting import (
    save_energy_plot,
    save_angle_plot,
)

In [19]:
def save_nll_plot(history, save_path):
    """
    Save the train and validation NLL curves across epochs.
    """
    hist_df = pd.DataFrame(history)

    plt.figure(figsize=(8, 5), dpi=130)
    plt.plot(hist_df["epoch"], hist_df["train_nll"], marker="o", label="Entrenamiento")
    plt.plot(hist_df["epoch"], hist_df["val_nll"], marker="o", label="Validación")

    plt.xlabel("Epoca")
    plt.ylabel("NLL")
    plt.title("Entrenamiento vs Validación NLL")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()

# Evaluation utilities for site-level and split-level assessment

The evaluation workflow used to assess the trained model on individual sites and dataset splits is defined. This block includes functions to evaluate each city, aggregate spectral errors across seeds, export metric tables, and generate comparison plots. For each site, the reference data are loaded, transformed to the target and context representation used during inference, filtered within the truncated normalized energy range, and compared against conditional model samples. The resulting energy and angular spectra are summarized using absolute relative error metrics, both globally and across predefined spectral ranges, and collected into Excel sheets for each evaluated split.

In [20]:
def evaluate_city(flow, city, data_base, stats, device, eval_seeds,
                  y_min=Y_MIN, y_max=Y_MAX, mu_min=MU_MIN, mu_max=MU_MAX,
                  sample_batch=SAMPLE_BATCH, bins_E=bins_E, bins_theta=bins_theta):
    """
    Evaluate the model on a single site and return summary metrics
    together with the data required for plotting.
    """
    # Load the site data and map all variables to the representation used in evaluation.
    df_city = pd.read_csv(data_base / f"{city}_3600.csv")
    df_city = logE_to_mm(df_city, stats)
    df_city = add_mu(df_city)
    df_city = con_mm(df_city, stats, context_cols)

    # Keep only rows inside the normalized energy interval used during truncated sampling.
    mask = df_city["logE_mm"].between(y_min, y_max)
    df_city = df_city.loc[mask].reset_index(drop=True)

    if len(df_city) == 0:
        raise ValueError(f"{city}: no rows after energy truncation.")

    # The conditioning vector is site-level, so one row is enough to define it.
    c_joint = torch.tensor(
        df_city.iloc[0][[c + "_mm" for c in context_cols]].to_numpy(np.float32),
        device=device
    ).view(1, -1)


    # Event counts after truncation
    n_events_arti = int(len(df_city))
    h_city = float(df_city["h"].iloc[0])
    n_events_cnf = int(expected_events_from_height(h_city, seconds=3600.0))
    rel_events = scalar_abs_relative_error(n_events_arti, n_events_cnf)

    E_real = df_city["energy"].to_numpy(dtype=np.float32)
    theta_real = df_city["theta"].to_numpy(dtype=np.float32)

    # Reference spectra from the evaluation data.
    bin_centers_E, spec_real_E = energy_spectrum(E_real, bins_E)
    bin_centers_A, spec_real_A = angle_spectrum(theta_real, bins_theta)

    # Per-seed outputs are stored first and aggregated afterwards.
    spec_models_E = []
    rank_errs_E = []
    plot_errs_E = []

    spec_models_A = []
    rank_errs_A = []
    plot_errs_A = []

    # Time only the inference stage.
    t0 = time.perf_counter()

    for s in eval_seeds:
        # Repeating the sampling with different seeds provides a stability estimate.
        seed_everything(int(s))

        y_joint = sample_model_joint(
            flow=flow,
            c=c_joint,
            y_min=y_min,
            y_max=y_max,
            mu_min=mu_min,
            mu_max=mu_max,
            batch=sample_batch,
            sample=n_events_cnf
        )

        y_logE_mm = y_joint[:, 0]
        y_mu = y_joint[:, 1]

        # Map sampled values back to the physical variables used for the spectra.
        E_samp = mm_to_energy(y_logE_mm, stats)
        theta_samp = mu_to_theta(y_mu)

        _, spec_model_E = energy_spectrum(E_samp, bins_E)
        _, spec_model_A = angle_spectrum(theta_samp, bins_theta)

        spec_models_E.append(spec_model_E)
        spec_models_A.append(spec_model_A)

        # Absolute relative errors are used for ranking and summary metrics.
        rank_errs_E.append(abs_relative_error_per_bin(spec_real_E, spec_model_E))
        plot_errs_E.append(weighted_relative_error_per_bin(spec_real_E, spec_model_E))

        rank_errs_A.append(abs_relative_error_per_bin(spec_real_A, spec_model_A))
        plot_errs_A.append(weighted_relative_error_per_bin(spec_real_A, spec_model_A))

    time_sec = float(time.perf_counter() - t0)

    spec_models_E = np.vstack(spec_models_E)
    rank_errs_E = np.vstack(rank_errs_E)
    plot_errs_E = np.vstack(plot_errs_E)

    spec_models_A = np.vstack(spec_models_A)
    rank_errs_A = np.vstack(rank_errs_A)
    plot_errs_A = np.vstack(plot_errs_A)

    # Aggregate the per-seed results into mean spectra and dispersion bands.
    spec_mean_E = np.mean(spec_models_E, axis=0)
    spec_std_E = np.std(spec_models_E, axis=0)
    rank_mean_E = np.nanmean(rank_errs_E, axis=0)
    plot_mean_E = np.nanmean(plot_errs_E, axis=0)
    plot_std_E = np.nanstd(plot_errs_E, axis=0)

    spec_mean_A = np.mean(spec_models_A, axis=0)
    spec_std_A = np.std(spec_models_A, axis=0)
    rank_mean_A = np.nanmean(rank_errs_A, axis=0)
    plot_mean_A = np.nanmean(plot_errs_A, axis=0)
    plot_std_A = np.nanstd(plot_errs_A, axis=0)

    # Summary metrics are reported globally and by predefined spectral intervals.
    energy_metrics = {
        "city": city,
        "rel_gl_mean": masked_mean(rank_mean_E),
        "rel_1e-1_1e2": range_mean(rank_mean_E, bin_centers_E, 1e-1, 1e2),
        "rel_1e2_1e3": range_mean(rank_mean_E, bin_centers_E, 1e2, 1e3),
        "rel_1e3_1e4": range_mean(rank_mean_E, bin_centers_E, 1e3, 1e4),
        "N_events_Arti": n_events_arti,
        "N_events_CNFs": n_events_cnf,
        "rel_events": rel_events,
        "time_sec": time_sec,
    }

    angle_metrics = {
        "city": city,
        "rel_gl_mean": masked_mean(rank_mean_A),
        "rel_0_30": range_mean(rank_mean_A, bin_centers_A, 0.0, 30.0),
        "rel_30_60": range_mean(rank_mean_A, bin_centers_A, 30.0, 60.0),
        "rel_60_90": range_mean(rank_mean_A, bin_centers_A, 60.0, 90.0 + 1e-9),
        "N_events_Arti": n_events_arti,
        "N_events_CNFs": n_events_cnf,
        "rel_events": rel_events,
        "time_sec": time_sec,
    }

    # Plot payload keeps only the arrays needed to build the evaluation figures.
    plot_payload = {
        "bin_centers_E": bin_centers_E,
        "spec_real_E": spec_real_E,
        "spec_mean_E": spec_mean_E,
        "spec_std_E": spec_std_E,
        "plot_mean_E": plot_mean_E,
        "plot_std_E": plot_std_E,
        "bin_centers_A": bin_centers_A,
        "spec_real_A": spec_real_A,
        "spec_mean_A": spec_mean_A,
        "spec_std_A": spec_std_A,
        "plot_mean_A": plot_mean_A,
        "plot_std_A": plot_std_A,
    }

    return energy_metrics, angle_metrics, plot_payload


def build_metric_total(metrics_energy_df, metrics_angle_df, split_name):
    """
    Build the aggregated total sheet for one evaluation split.
    """
    rel_gl_mean_energy = float(metrics_energy_df["rel_gl_mean"].mean())
    rel_gl_mean_angle = float(metrics_angle_df["rel_gl_mean"].mean())
    rel_gl_mean_total = float((rel_gl_mean_energy + rel_gl_mean_angle) / 2.0)

    # Event counts, rel_events and time_sec are the same in both sheets.
    n_events_arti_total = int(metrics_energy_df["N_events_Arti"].sum())
    n_events_cnf_total = int(metrics_energy_df["N_events_CNFs"].sum())
    rel_events_mean = float(metrics_energy_df["rel_events"].mean())
    time_sec_total = float(metrics_energy_df["time_sec"].sum())

    return pd.DataFrame([{
        "split": split_name,
        "rel_gl_mean_energy": rel_gl_mean_energy,
        "rel_gl_mean_angle": rel_gl_mean_angle,
        "rel_gl_mean_total": rel_gl_mean_total,
        "N_events_Arti": n_events_arti_total,
        "N_events_CNFs": n_events_cnf_total,
        "rel_events": rel_events_mean,
        "time_sec": time_sec_total,
    }])


def save_metrics_workbook(metrics_energy_df, metrics_angle_df, metrics_total_df, save_path):
    """
    Save the metric tables into a single Excel workbook.
    """
    with pd.ExcelWriter(save_path) as writer:
        metrics_energy_df.to_excel(writer, sheet_name="metric_energy", index=False)
        metrics_angle_df.to_excel(writer, sheet_name="metric_angle", index=False)
        metrics_total_df.to_excel(writer, sheet_name="metric_total", index=False)


def evaluate_dataset(flow, split_name, sites, data_base, stats, device, output_dir, images_dir):
    """
    Evaluate the model on all sites of a split, save plots and metric tables,
    and return the aggregated summary.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    images_dir.mkdir(parents=True, exist_ok=True)

    rows_energy = []
    rows_angle = []

    for city in sites:
        print(f"[{split_name}] Evaluating {city} ...")

        energy_metrics, angle_metrics, plot_payload = evaluate_city(
            flow=flow,
            city=city,
            data_base=data_base,
            stats=stats,
            device=device,
            eval_seeds=EVAL_SEEDS
        )

        rows_energy.append(energy_metrics)
        rows_angle.append(angle_metrics)

        # Save one energy figure and one angular figure per evaluated city.
        save_energy_plot(
            city=city,
            bin_centers_E=plot_payload["bin_centers_E"],
            spec_real_E=plot_payload["spec_real_E"],
            spec_mean_E=plot_payload["spec_mean_E"],
            spec_std_E=plot_payload["spec_std_E"],
            contrib_mean_E=plot_payload["plot_mean_E"],
            contrib_std_E=plot_payload["plot_std_E"],
            save_path=images_dir / f"{city}_energy.png"
        )

        save_angle_plot(
            city=city,
            bin_centers_A=plot_payload["bin_centers_A"],
            spec_real_A=plot_payload["spec_real_A"],
            spec_mean_A=plot_payload["spec_mean_A"],
            spec_std_A=plot_payload["spec_std_A"],
            contrib_mean_A=plot_payload["plot_mean_A"],
            contrib_std_A=plot_payload["plot_std_A"],
            save_path=images_dir / f"{city}_angle.png"
        )

    metrics_energy_df = pd.DataFrame(rows_energy)
    metrics_angle_df = pd.DataFrame(rows_angle)
    metrics_total_df = build_metric_total(metrics_energy_df, metrics_angle_df, split_name)

    xlsx_path = output_dir / "metrics.xlsx"
    save_metrics_workbook(metrics_energy_df, metrics_angle_df, metrics_total_df, xlsx_path)

    return {
        "metrics_energy_df": metrics_energy_df,
        "metrics_angle_df": metrics_angle_df,
        "metrics_total_df": metrics_total_df,
        "xlsx_path": xlsx_path,
    }

In [21]:
def _json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_json_default)

# Final training and evaluation workflow

The final workflow for model training, evaluation, and result export is defined. This block includes the preparation of the required metadata, the training of the selected model configuration, the evaluation on the corresponding dataset splits, and the storage of the generated outputs.

In [ ]:
split_sites = {
    "train_sites": train_sites,
    "val_sites": val_sites,
    "test_sites": test_sites,
}

save_json(MODEL_FINAL_DIR / "split_sites.json", split_sites)
save_json(MODEL_FINAL_DIR / "stats.json", stats)

# Create data loaders
train_loader = make_loader(
    train_tensor,
    batch_size=4096,
    shuffle=True,
    num_workers=2
)

val_loader = make_loader(
    val_tensor,
    batch_size=8192,
    shuffle=False,
    num_workers=2
)

print("=" * 90)
print("FINAL MODEL CONFIG")
print({
    "base_seed": BASE_SEED,
    "lr": LR_FIXED,
    "epochs": EPOCHS,
    "hidden_features": FINAL_HIDDEN_FEATURES,
    "num_layers": FINAL_NUM_LAYERS,
    "num_bins": FINAL_NUM_BINS,
    "tail_bound": FINAL_TAIL_BOUND,
    "eval_seeds": EVAL_SEEDS,
    "y_min": Y_MIN,
    "y_max": Y_MAX,
    "mu_min": MU_MIN,
    "mu_max": MU_MAX,
    "sample_batch": SAMPLE_BATCH,
})

# Reset seeds before training
reset_for_trial(BASE_SEED)

# Build and train the model
flow = build_flow(
    hidden_features=FINAL_HIDDEN_FEATURES,
    num_layers=FINAL_NUM_LAYERS,
    num_bins=FINAL_NUM_BINS,
    tail_bound=FINAL_TAIL_BOUND
)

flow, history = train_flow(
    flow=flow,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=EPOCHS,
    lr=LR_FIXED
)

# Save trained weights
torch.save(flow.state_dict(), MODEL_FINAL_DIR / "model.pt")

# Save training history
save_json(MODEL_FINAL_DIR / "history.json", history)

# Save NLL curve
save_nll_plot(history, MODEL_FINAL_DIR / "nll_curve.png")

# Save final configuration
config = {
    "base_seed": BASE_SEED,
    "epochs": EPOCHS,
    "lr": LR_FIXED,
    "hidden_features": FINAL_HIDDEN_FEATURES,
    "num_layers": FINAL_NUM_LAYERS,
    "num_bins": FINAL_NUM_BINS,
    "tail_bound": FINAL_TAIL_BOUND,
    "eval_seeds": EVAL_SEEDS,
    "y_min": Y_MIN,
    "y_max": Y_MAX,
    "mu_min": MU_MIN,
    "mu_max": MU_MAX,
    "sample_batch": SAMPLE_BATCH,
    "target_per_site_train": TARGET_PER_SITE_TRAIN,
    "n_bins_downsample": N_BINS,
    "event_formula": "N_events = 3600 * 101 * exp(0.0002109 * h)",
}
save_json(MODEL_FINAL_DIR / "config.json", config)


# Evaluate final model
flow = flow.to(device).eval()

# Validation split
val_out = evaluate_dataset(
    flow=flow,
    split_name="val",
    sites=val_sites,
    data_base=DATA_BASE,
    stats=stats,
    device=device,
    output_dir=VAL_DIR,
    images_dir=VAL_IMG_DIR
)

# Test split
test_out = evaluate_dataset(
    flow=flow,
    split_name="test",
    sites=test_sites,
    data_base=DATA_BASE,
    stats=stats,
    device=device,
    output_dir=TEST_DIR,
    images_dir=TEST_IMG_DIR
)

# Save summary
summary = {
    "val_metric_total": val_out["metrics_total_df"].to_dict(orient="records"),
    "test_metric_total": test_out["metrics_total_df"].to_dict(orient="records"),
}
save_json(MODEL_FINAL_DIR / "summary.json", summary)

A compressed .zip archive is created containing the full output directory of the final model workflow. Packaging all generated files into a single archive simplifies download, storage, and later access to the trained model, evaluation tables, summary files, and spectral comparison figures.

In [ ]:
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(ZIP_PATH.with_suffix("")),
    format="zip",
    root_dir=MODEL_FINAL_DIR.parent,
    base_dir=MODEL_FINAL_DIR.name
)

print("Final model finished.")
print("Folder:", MODEL_FINAL_DIR)
print("Validation workbook:", VAL_DIR / "metrics.xlsx")
print("Test workbook:", TEST_DIR / "metrics.xlsx")
print("ZIP:", ZIP_PATH)